# Claims Data Quality Profiling

## Purpose

In this notebook, I profile the Silver claims dataset and define the
data-quality rules that will later be attached directly to the Lakeflow
Bronze-to-Silver transformation.

I am not creating another transformed claims table here.

The purpose of this stage is to:

- identify invalid or incomplete records
- measure current rule violations
- classify rules as warning, drop, or fail
- document the quality contract
- prepare reusable Lakeflow expectation definitions
- puplish them to the helthy_insurance.governmence.quality_rules

### Source

`health_insurance.silver.claims`

### Production design

The final production flow will be:

Bronze  
↓  
Silver transformation  
+  
Lakeflow expectations  
↓  
Validated Silver  
↓  
Gold



In [0]:
# loading the current Silver claims dataset for quality profiling.

from pyspark.sql import functions as F

CLAIMS_TABLE = "health_insurance.silver.claims"

claims_df = spark.table(CLAIMS_TABLE)

print(f"Claims rows: {claims_df.count():,}")

display(claims_df.limit(10))

In [0]:
# defining the claims quality contract by severity. rather than scattering rules throughout the code, we define them here as a dictionary.



CLAIMS_WARN_RULES = {
    "known_patient_gender":
        "patient_gender IN ('M', 'F')",

    "service_type_present":
        "service_type IS NOT NULL",

    "provider_specialty_present":
        "provider_specialty IS NOT NULL"
}


CLAIMS_DROP_RULES = {
    "claim_id_present":
        "claim_id IS NOT NULL",

    "patient_id_present":
        "patient_id IS NOT NULL",

    "policy_number_present":
        "policy_number IS NOT NULL"
}


CLAIMS_FAIL_RULES = {
    "claim_amount_non_negative":
        "claim_amount >= 0",

    "patient_age_valid":
        "patient_age BETWEEN 0 AND 120",

    "service_not_after_claim":
        "service_date <= claim_date"
}

In [0]:
# measuring how many rows violate each quality rule.

def profile_rules(df, rules, severity):
    results = []

    total_rows = df.count()

    for rule_name, condition in rules.items():

        failed_rows = (
            df
            .filter(f"NOT ({condition}) OR ({condition}) IS NULL")
            .count()
        )

        results.append(
            (
                rule_name,
                severity,
                condition,
                total_rows,
                failed_rows,
                round(
                    failed_rows / total_rows * 100,
                    2
                ) if total_rows else 0.0
            )
        )

    return results

In [0]:
quality_results = []

quality_results += profile_rules(
    claims_df,
    CLAIMS_WARN_RULES,
    "WARN"
)

quality_results += profile_rules(
    claims_df,
    CLAIMS_DROP_RULES,
    "DROP"
)

quality_results += profile_rules(
    claims_df,
    CLAIMS_FAIL_RULES,
    "FAIL"
)

In [0]:
# presenting the claims quality profile as a structured result.

quality_profile_df = spark.createDataFrame(
    quality_results,
    [
        "rule_name",
        "severity",
        "constraint",
        "total_rows",
        "failed_rows",
        "failed_percentage"
    ]
)

display(
    quality_profile_df
    .orderBy(
        "severity",
        F.desc("failed_percentage")
    )
)

In [0]:
# frominspection the gender ruleis giving a 100% failour rate so now i am inspecting the distinct standardized gender values
# before finalizing the quality rule.

claims_df.select(
    "patient_gender"
).distinct().orderBy(
    "patient_gender"
).show()

in the inspection i realized the the values are supposed to be MALE, FEMALE AND OTHER so i will now re difine the rule

In [0]:
# final rule definitions 

CLAIMS_WARN_RULES = {
    "known_patient_gender":
        "patient_gender IN ('MALE', 'FEMALE', 'OTHER')",

    "service_type_present":
        "service_type IS NOT NULL",

    "provider_specialty_present":
        "provider_specialty IS NOT NULL"
}
CLAIMS_DROP_RULES = {
    "claim_id_present":
        "claim_id IS NOT NULL",

    "patient_id_present":
        "patient_id IS NOT NULL",

    "policy_number_present":
        "policy_number IS NOT NULL",

    "patient_age_valid":
        "patient_age BETWEEN 0 AND 120"
}
CLAIMS_FAIL_RULES = {
    "claim_amount_non_negative":
        "claim_amount >= 0",

    "service_not_after_claim":
        "service_date <= claim_date"
}

## Publish approved Claims quality rules

I have completed the Claims quality profiling and finalized the approved
WARN, DROP, and FAIL rules.

I now publish these rules directly into the central Unity Catalog governance
table so the Lakeflow pipeline can retrieve them dynamically.

This keeps the rule definition connected to the notebook where I developed
and validated it.

In [0]:
# converting the finalized Claims quality contract
# into rows for the central governance repository.

from pyspark.sql import functions as F

def build_rule_rows(dataset, severity, rules, description, source_notebook):

    return [
        (
            dataset,
            rule_name,
            constraint.strip(),
            severity,
            True,
            description,
            source_notebook,
            "data_engineering",
            1
        )
        for rule_name, constraint in rules.items()
    ]

In [0]:
# preparing all approved Claims rules for publication.

claims_rule_rows = []

claims_rule_rows += build_rule_rows(
    "claims",
    "WARN",
    CLAIMS_WARN_RULES,
    "Claims quality monitoring rule",
    "04-data-quality/01_claims_quality_profile"
)

claims_rule_rows += build_rule_rows(
    "claims",
    "DROP",
    CLAIMS_DROP_RULES,
    "Claims record validity rule",
    "04-data-quality/01_claims_quality_profile"
)

claims_rule_rows += build_rule_rows(
    "claims",
    "FAIL",
    CLAIMS_FAIL_RULES,
    "Critical Claims pipeline rule",
    "04-data-quality/01_claims_quality_profile"
)

In [0]:
#  creating the Claims quality-rule publication DataFrame.

claims_rules_df = (
    spark.createDataFrame(
        claims_rule_rows,
        [
            "dataset",
            "rule_name",
            "constraint",
            "severity",
            "is_active",
            "description",
            "source_notebook",
            "owner",
            "version"
        ]
    )
    .withColumn(
        "created_at",
        F.current_timestamp()
    )
    .withColumn(
        "updated_at",
        F.current_timestamp()
    )
)

display(claims_rules_df)

In [0]:
# exposing the finalized Claims rules
# as a temporary view for idempotent publishing.

claims_rules_df.createOrReplaceTempView(
    "claims_quality_rule_updates"
)

In [0]:
%sql
-- publishing the approved Claims rules
-- directly from this profiling notebook.

MERGE INTO health_insurance.governance.quality_rules AS target

USING claims_quality_rule_updates AS source

ON target.dataset = source.dataset
AND target.rule_name = source.rule_name

WHEN MATCHED THEN UPDATE SET

    target.constraint = source.constraint,
    target.severity = source.severity,
    target.is_active = source.is_active,
    target.description = source.description,
    target.source_notebook = source.source_notebook,
    target.owner = source.owner,

    target.version =
        CASE
            WHEN target.constraint <> source.constraint
              OR target.severity <> source.severity
            THEN COALESCE(target.version, 1) + 1
            ELSE target.version
        END,

    target.updated_at = source.updated_at

WHEN NOT MATCHED THEN INSERT (
    dataset,
    rule_name,
    constraint,
    severity,
    is_active,
    description,
    source_notebook,
    owner,
    version,
    created_at,
    updated_at
)

VALUES (
    source.dataset,
    source.rule_name,
    source.constraint,
    source.severity,
    source.is_active,
    source.description,
    source.source_notebook,
    source.owner,
    source.version,
    source.created_at,
    source.updated_at
);

In [0]:
%sql
-- verifying the Claims rules published by this notebook.

SELECT
    dataset,
    rule_name,
    severity,
    constraint,
    source_notebook,
    version,
    is_active,
    updated_at
FROM health_insurance.governance.quality_rules
WHERE dataset = 'claims'
ORDER BY severity, rule_name;